# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [45]:
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [46]:
MODEL_LLAMA = "llama3.2"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [57]:
system_prompt = """
                    You are the top living expert in the Python world, extremely Pythonic, and able to answer with 
                    accuracy and authority any question or request about code, best practices, and frameworks 
                    related to Python, across all its versions—from the very first to the latest.
                """

user_prompt = """
                    Please explain what this code does and why:
                    yield from (book['author'] for book in books if 'author' in book)
              """

In [58]:
def answer_question(system_prompt, user_prompt, model_client, model_name):
    stream = model_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [59]:
answer_question(system_prompt, user_prompt, ollama, MODEL_LLAMA) 

**Code Explanation**

This code is using a combination of Python 3.3+ features, specifically `yield from` and dictionary iteration.

```python
yield from (book['author'] for book in books if 'author' in book)
```

Here's what it does:

1. **List comprehension**: `(book['author'] for book in books if 'author' in book)` is a list comprehension that:
	* Iterates over the `books` dictionary.
	* Filters out books where `'author'` is not present (using `if 'author' in book`).
	* For each book with an `'author'`, yields its `'author'` value.
2. **`yield from`**: The entire expression `(book['author'] for book in books if 'author' in book)` is passed to the `yield from` keyword.
	* `yield from` takes a single iterator (in this case, an iterable created by the list comprehension) and yields each item produced by that iterator.
3. **Yielding**: The `yield from` expression effectively "unwraps" the top-level iteration, replacing it with the innermost iteration.

**What it does in practice**

When you run this code, it will:

1. Iterate over the `books` dictionary.
2. For each book, check if `'author'` is present (i.e., its value is a list or another dictionary without `'author'`).
3. If `'author'` is present, yield its value.

**Why and when to use it**

You'd use this code:

* When you need to iterate over specific values within another data structure (like `books`), but instead of directly iterating over that structure, you want to process the values in a more controlled manner.
* To simplify complex iteration logic or chain multiple conditions together.

Example usage:

```python
books = {
    'book1': {'author': 'Author 1', 'title': 'Title 1'},
    'book2': {'author': 'Author 2', 'title': 'Title 2'},
    'book3': {'publisher': 'Publisher 3'}
}

for author in yield from (book['author'] for book in books if 'author' in book):
    print(author)  # Output: Author 1, Author 2

for value in yield from (book.get('other_key', None) for book in books.values()):
    print(value)

# You could even write it as:

import itertools

def filtered_values(sequence, filter_function=lambda x: True):
    return tuple(filter(filter_function, sequence))

filtered_books = filtered_values(books, lambda book: 'author' in book)
for author in yield from filtered_books:
    print(author)  # Output: Author 1, Author 2
```

This shows how `yield from` can simplify complex iteration and filter logic into a more Pythonic and readable form.